# 03 — 特征工程（PySpark）

**负责人：Alex**  
**前提：** `01_data_download.ipynb` 已运行，`robot_frames.parquet` 已存到 Google Drive

**目标：** 从 251,802 帧中提取 30 个纯运动学特征（不含 reward），聚合到 episode 级

## 特征设计原则

1. **无 reward 泄露**：特征全部来自运动轨迹（state/action），不使用任何 reward 信息
2. **无 time_efficiency**：不用 `-episode_length` 作特征（真实机器人的质量标签用 episode_length 定义，直接用会循环）
3. **维度无关**：用 L2 模长聚合，兼容 pusht(2D) / xarm(4D) / real(8D) 不同维度

## 两套质量标签（实验设计）

| 数据集类型 | 标签名 | 定义 | 原因 |
|-----------|--------|------|------|
| 仿真（5个）| `reward_quality` | `max_reward > per-dataset 中位数` | 有连续 reward 信号 |
| 真实机器人（3个）| `efficiency_quality` | `episode_length < per-dataset 中位数` | 全部 max_reward=1（都成功），用效率区分 |

In [1]:
!pip install pyspark pyarrow pandas numpy -q
print('依赖安装完成')

依赖安装完成


In [2]:
import os, math
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/bigdata-project'
except ImportError:
    DATA_DIR = '../data'

frames = pd.read_parquet(f'{DATA_DIR}/robot_frames.parquet')

# !! 关键：parquet 读出的 list 列可能是 ndarray，Spark 不支持，必须转 list
frames['observation_state'] = frames['observation_state'].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else list(x))
frames['action'] = frames['action'].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else list(x))

print(f'帧级数据: {frames.shape}')
print(f'observation_state 类型: {type(frames["observation_state"].iloc[0])}')
frames.head(2)

帧级数据: (251802, 7)
observation_state 类型: <class 'list'>


,source,episode_index,frame_index,observation_state,action,next_reward,next_done
0,pusht,0,0,"[222.0, 97.0]","[233.0, 71.0]",0.190297,False
1,pusht,0,1,"[225.2523956298828, 89.31253051757812]","[229.0, 83.0]",0.190297,False


In [3]:
# this cell is added by Jinqiang Ding for something important 

import os
import sys

print("当前 notebook Python:", sys.executable)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

当前 notebook Python: c:\Users\ding\anaconda3\envs\bigdata-project\python.exe


In [4]:
# changed by Jinqiang Ding for something important   


from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bigdata-project")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark 版本:", spark.version)
print("Driver Python:", sys.executable)
print("Spark master:", sc.master)

Spark 版本: 3.5.1
Driver Python: c:\Users\ding\anaconda3\envs\bigdata-project\python.exe
Spark master: local[2]


In [5]:
# changed by Jinqiang Ding for something important 

from pyspark.sql.functions import udf, col
from pyspark.sql.types import *
import math
import pyspark.sql.functions as F
from pyspark.sql.window import Window


# 转为 Spark DataFrame
sdf = spark.createDataFrame(frames)
print(f'Spark DataFrame: {sdf.count():,} 行')

# 计算帧级 L2 模长
@udf(returnType=DoubleType())
def l2_norm(vec):
    if vec is None: return 0.0
    return float(math.sqrt(sum(v * v for v in vec)))

sdf = sdf.withColumn('state_norm',  l2_norm(col('observation_state'))) \
         .withColumn('action_norm', l2_norm(col('action')))
sdf.select('source','episode_index','frame_index','state_norm','action_norm').show(5)

Spark DataFrame: 251,802 行
+------+-------------+-----------+------------------+------------------+
|source|episode_index|frame_index|        state_norm|       action_norm|
+------+-------------+-----------+------------------+------------------+
| pusht|            0|          0|242.26638231500465|243.57750306627253|
| pusht|            0|          1| 242.3125457842713|243.57750306627253|
| pusht|            0|          2|242.78453445790046| 244.6160256401857|
| pusht|            0|          3|243.47251854915743|245.55243839147678|
| pusht|            0|          4| 244.2909082595288|255.03333115496883|
+------+-------------+-----------+------------------+------------------+
only showing top 5 rows



In [6]:
# Episode 级聚合（PySpark 内完成）
ep_agg = sdf.groupBy('source', 'episode_index').agg(
    F.count('*').alias('episode_length'),
    F.max('next_reward').alias('max_reward'),
    F.avg('next_reward').alias('avg_reward'),
    F.avg('state_norm').alias('state_mean'),
    F.stddev_pop('state_norm').alias('state_std'),
    F.min('state_norm').alias('state_min'),
    F.max('state_norm').alias('state_max'),
    F.avg('action_norm').alias('action_mean'),
    F.stddev_pop('action_norm').alias('action_std'),
    F.min('action_norm').alias('action_min'),
    F.max('action_norm').alias('action_max'),
    # 收集时序序列（用于复杂特征计算）
    F.sort_array(
        F.collect_list(F.struct('frame_index', 'state_norm', 'action_norm'))
    ).alias('seq'),
)
print(f'Episode 数: {ep_agg.count():,}')

Episode 数: 5,026


In [7]:
# 32 个运动学特征（UDF 实现）
# 文献来源：
#   - Luo et al. (2024) "Consistency Matters: Defining Demonstration Data Quality Metrics"
#     发现 path_length 和 jerk 是预测学习效果最强的指标
#   - Mandlekar et al. (CoRL 2021) "What Matters in Learning from Offline Human Demonstrations"
#     研究不同质量演示数据对策略训练的影响
RichSchema = StructType([
    StructField('state_range',          DoubleType()),
    StructField('action_range',         DoubleType()),
    StructField('smoothness',           DoubleType()),
    StructField('path_efficiency',      DoubleType()),
    StructField('action_consistency',   DoubleType()),
    StructField('correction_count',     DoubleType()),
    StructField('state_diversity',      DoubleType()),
    StructField('final_stability',      DoubleType()),
    StructField('action_jerk_mean',     DoubleType()),
    StructField('action_jerk_max',      DoubleType()),
    StructField('velocity_mean',        DoubleType()),
    StructField('velocity_std',         DoubleType()),
    StructField('velocity_max',         DoubleType()),
    StructField('state_trend',          DoubleType()),
    StructField('action_trend',         DoubleType()),
    StructField('state_autocorr',       DoubleType()),
    StructField('action_autocorr',      DoubleType()),
    StructField('early_action_mean',    DoubleType()),
    StructField('late_action_mean',     DoubleType()),
    StructField('action_warmup_ratio',  DoubleType()),
    StructField('state_net_change',     DoubleType()),
    StructField('state_final_deviation',DoubleType()),
    # 文献支持的特征（Luo et al. 2024）
    StructField('total_path_length',    DoubleType()),  # 累计路径长度（state 空间）
    StructField('action_effort',        DoubleType()),  # 动作模长平方均值（能量代理）
])

@udf(returnType=RichSchema)
def compute_rich_features(seq):
    if seq is None or len(seq) < 3:
        return tuple([0.0] * 24)
    seq = sorted(seq, key=lambda r: r['frame_index'])
    sn  = [float(f['state_norm'])  for f in seq]
    an  = [float(f['action_norm']) for f in seq]
    n   = len(sn)

    state_range = max(sn) - min(sn)
    action_range = max(an) - min(an)

    diffs_a = [abs(an[i+1]-an[i]) for i in range(n-1)]
    smoothness = -(sum(diffs_a)/len(diffs_a)) if diffs_a else 0.0

    total_path = sum(abs(sn[i+1]-sn[i]) for i in range(n-1))
    path_efficiency = abs(sn[-1]-sn[0])/total_path if total_path > 1e-6 else 1.0

    mean_a = sum(an)/n
    action_consistency = -math.sqrt(sum((a-mean_a)**2 for a in an)/n)

    reversals = sum(1 for i in range(1,n-1)
                    if (sn[i]-sn[i-1])*(sn[i+1]-sn[i]) < 0)
    correction_count = reversals/n

    mean_s = sum(sn)/n
    state_diversity = math.sqrt(sum((s-mean_s)**2 for s in sn)/n)

    tail = sn[-(max(3,n//5)):]
    mean_t = sum(tail)/len(tail)
    final_stability = -math.sqrt(sum((s-mean_t)**2 for s in tail)/len(tail))

    jerk = [abs(an[i+2]-2*an[i+1]+an[i]) for i in range(n-2)] if n >= 3 else [0.0]
    action_jerk_mean = sum(jerk)/len(jerk)
    action_jerk_max  = max(jerk)

    vel = [abs(sn[i+1]-sn[i]) for i in range(n-1)] if n >= 2 else [0.0]
    velocity_mean = sum(vel)/len(vel)
    mean_v = velocity_mean
    velocity_std  = math.sqrt(sum((v-mean_v)**2 for v in vel)/len(vel))
    velocity_max  = max(vel)

    t = list(range(n))
    def linslope(xs, ys):
        mx = sum(xs)/n; my = sum(ys)/n
        num = sum((x-mx)*(y-my) for x,y in zip(xs,ys))
        den = sum((x-mx)**2 for x in xs)
        return num/den if den > 1e-8 else 0.0
    state_trend  = linslope(t, sn)
    action_trend = linslope(t, an)

    def autocorr(xs):
        m = sum(xs)/len(xs)
        num = sum((xs[i]-m)*(xs[i+1]-m) for i in range(len(xs)-1))
        den = sum((x-m)**2 for x in xs)
        return num/den if den > 1e-8 else 0.0
    state_autocorr  = autocorr(sn)
    action_autocorr = autocorr(an)

    q = max(2, n//4)
    early_action_mean = sum(an[:q])/q
    late_action_mean  = sum(an[-q:])/q
    warmup = late_action_mean/early_action_mean if early_action_mean > 1e-8 else 1.0

    state_net_change      = abs(sn[-1]-sn[0])
    state_final_deviation = abs(sn[-1]-mean_s)

    # 文献支持的特征
    total_path_length = total_path                     # 累计路径长度（Luo et al. 2024）
    action_effort     = sum(a*a for a in an) / n       # 能量代理（关节力矩平方和均值）

    return (state_range, action_range, smoothness, path_efficiency,
            action_consistency, correction_count, state_diversity, final_stability,
            action_jerk_mean, action_jerk_max, velocity_mean, velocity_std, velocity_max,
            state_trend, action_trend, state_autocorr, action_autocorr,
            early_action_mean, late_action_mean, warmup,
            state_net_change, state_final_deviation,
            total_path_length, action_effort)

print('UDF 定义完成（32 个运动学特征）')

UDF 定义完成（32 个运动学特征）


In [8]:
# 应用 UDF + 展开
rich_fields = [
    'state_range','action_range','smoothness','path_efficiency',
    'action_consistency','correction_count','state_diversity','final_stability',
    'action_jerk_mean','action_jerk_max','velocity_mean','velocity_std','velocity_max',
    'state_trend','action_trend','state_autocorr','action_autocorr',
    'early_action_mean','late_action_mean','action_warmup_ratio',
    'state_net_change','state_final_deviation',
    # 文献支持（Luo et al. 2024）
    'total_path_length','action_effort',
]

ep_feat = ep_agg.withColumn('rf', compute_rich_features(col('seq'))).select(
    'source','episode_index','episode_length',
    'max_reward','avg_reward',
    'state_mean','state_std','state_min','state_max',
    'action_mean','action_std','action_min','action_max',
    *[col(f'rf.{f}').alias(f) for f in rich_fields],
)
print(f'特征 DataFrame: {ep_feat.count():,} episodes × {len(ep_feat.columns)} 列')
print(f'纯运动学特征数: {len(rich_fields) + 8} (rich + basic stats)')

特征 DataFrame: 5,026 episodes × 37 列
纯运动学特征数: 32 (rich + basic stats)


In [9]:
# 双标签定义
REAL_SOURCES = [
    'berkeley_autolab_ur5',
    'columbia_cairlab_pusht_real',
    'nyu_door_opening_surprising_effectiveness',
]

window = Window.partitionBy('source')

# reward_quality: 仿真数据用
ep_feat = ep_feat.withColumn(
    'reward_median', F.percentile_approx('max_reward', 0.5).over(window)
).withColumn(
    'reward_quality',
    (F.col('max_reward') > F.col('reward_median')).cast('integer')
)

# efficiency_quality: 真实机器人数据用（episode 越短 = 越高效 = 高质量）
ep_feat = ep_feat.withColumn(
    'length_median', F.percentile_approx('episode_length', 0.5).over(window)
).withColumn(
    'efficiency_quality',
    (F.col('episode_length') < F.col('length_median')).cast('integer')
)

print('标签分布：')
ep_feat.groupBy('source','reward_quality','efficiency_quality').count() \
       .orderBy('source').show(20)

标签分布：
+--------------------+--------------+------------------+-----+
|              source|reward_quality|efficiency_quality|count|
+--------------------+--------------+------------------+-----+
|berkeley_autolab_ur5|             0|                 0|  508|
|berkeley_autolab_ur5|             0|                 1|  492|
|columbia_cairlab_...|             0|                 0|   69|
|columbia_cairlab_...|             0|                 1|   67|
|nyu_door_opening_...|             0|                 0|  255|
|nyu_door_opening_...|             0|                 1|  229|
|               pusht|             1|                 0|   61|
|               pusht|             1|                 1|   42|
|               pusht|             0|                 1|   60|
|               pusht|             0|                 0|   43|
|    xarm_lift_medium|             0|                 0|  401|
|    xarm_lift_medium|             1|                 0|  399|
|xarm_lift_medium_...|             0|            

In [10]:
# 保存
ep_pd = ep_feat.toPandas()

save_path = f'{DATA_DIR}/episode_features_rich.parquet'
ep_pd.to_parquet(save_path, index=False)

size_mb = os.path.getsize(save_path) / 1024 / 1024
print(f'保存: {save_path}')
print(f'形状: {ep_pd.shape}  大小: {size_mb:.1f} MB')
print(f'\n特征列 ({len(ep_pd.columns)} 列):')
for c in ep_pd.columns:
    print(f'  {c}')

spark.stop()
print('\n✅ 特征工程完成！下一步: 04_modeling.ipynb')

保存: ../data/episode_features_rich.parquet
形状: (5026, 41)  大小: 1.5 MB

特征列 (41 列):
  source
  episode_index
  episode_length
  max_reward
  avg_reward
  state_mean
  state_std
  state_min
  state_max
  action_mean
  action_std
  action_min
  action_max
  state_range
  action_range
  smoothness
  path_efficiency
  action_consistency
  correction_count
  state_diversity
  final_stability
  action_jerk_mean
  action_jerk_max
  velocity_mean
  velocity_std
  velocity_max
  state_trend
  action_trend
  state_autocorr
  action_autocorr
  early_action_mean
  late_action_mean
  action_warmup_ratio
  state_net_change
  state_final_deviation
  total_path_length
  action_effort
  reward_median
  reward_quality
  length_median
  efficiency_quality

✅ 特征工程完成！下一步: 04_modeling.ipynb
